# Part 1 — MCQ (Q1–Q10)

Notebook giải toàn bộ 10 câu trắc nghiệm trực tiếp từ dữ liệu.  
Mỗi câu: code → kết quả numeric → đáp án A/B/C/D.

**Issues:**
- [#3] TV1: Q1, Q5, Q8, Q10
- [#4] TV2: Q2, Q3, Q6, Q9
- [#7] Shared: Q4, Q7

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../data')

orders      = pd.read_csv(DATA / 'orders.csv',      parse_dates=['order_date'])
order_items = pd.read_csv(DATA / 'order_items.csv')
payments    = pd.read_csv(DATA / 'payments.csv')
products    = pd.read_csv(DATA / 'products.csv')
customers   = pd.read_csv(DATA / 'customers.csv')
returns     = pd.read_csv(DATA / 'returns.csv')
geography   = pd.read_csv(DATA / 'geography.csv')
sales       = pd.read_csv(DATA / 'sales.csv',       parse_dates=['Date'])
web_traffic = pd.read_csv(DATA / 'web_traffic.csv')

C:\Users\ASUS\AppData\Local\Temp\ipykernel_10844\1434424597.py:8: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(DATA / 'order_items.csv')


---
## Q1 — Trung vị inter-order gap (ngày) của khách hàng mua >1 lần

> Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
> mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)
>
> A) 30 ngày  B) 90 ngày  C) 180 ngày  D) 365 ngày

In [2]:
# Keep only customers with >1 order
repeat = orders.groupby('customer_id').filter(lambda g: len(g) > 1)

# Sort and compute consecutive gaps per customer
repeat_sorted = repeat.sort_values(['customer_id', 'order_date'])
gaps = (
    repeat_sorted
    .groupby('customer_id')['order_date']
    .diff()
    .dropna()
    .dt.days
)

median_gap = gaps.median()
print(f'Median inter-order gap: {median_gap:.1f} days')
print(f'  → A=30 | B=90 | C=180 | D=365')

Median inter-order gap: 144.0 days
  → A=30 | B=90 | C=180 | D=365


---
## Q2 — Segment có tỷ suất lợi nhuận gộp trung bình cao nhất

> Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
> trung bình cao nhất, với công thức (price − cogs)/price?
>
> A) Premium  B) Performance  C) Activewear  D) Standard

In [3]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

margin_by_segment = (
    products.groupby('segment')['gross_margin']
    .mean()
    .sort_values(ascending=False)
)
print(margin_by_segment.round(4).to_string())
print(f'\nBest segment: {margin_by_segment.idxmax()}')
print(f'  → A=Premium | B=Performance | C=Activewear | D=Standard')

segment
Standard       0.3134
Premium        0.2854
All-weather    0.2842
Activewear     0.2656
Performance    0.2636
Balanced       0.2580
Trendy         0.2408
Everyday       0.2363

Best segment: Standard
  → A=Premium | B=Performance | C=Activewear | D=Standard


---
## Q3 — Lý do trả hàng phổ biến nhất của Streetwear

> Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear
> (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?
>
> A) defective  B) wrong_size  C) changed_mind  D) not_as_described

In [4]:
returns_with_cat = returns.merge(
    products[['product_id', 'category']], on='product_id', how='left'
)

streetwear_returns = returns_with_cat[returns_with_cat['category'] == 'Streetwear']

reason_counts = streetwear_returns['return_reason'].value_counts()
print(reason_counts.to_string())
print(f'\nMost common: {reason_counts.idxmax()} ({reason_counts.max():,})')
print(f'  → A=defective | B=wrong_size | C=changed_mind | D=not_as_described')

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159

Most common: wrong_size (7,626)
  → A=defective | B=wrong_size | C=changed_mind | D=not_as_described


---
## Q4 — Traffic source có bounce_rate trung bình thấp nhất

> Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình
> (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó?
>
> A) organic_search  B) paid_search  C) email_campaign  D) social_media

In [5]:
bounce_by_source = (
    web_traffic.groupby('traffic_source')['bounce_rate']
    .mean()
    .sort_values()
)
print(bounce_by_source.round(4).to_string())
print(f'\nLowest bounce_rate: {bounce_by_source.idxmin()} ({bounce_by_source.min():.4f})')
print(f'  → A=organic_search | B=paid_search | C=email_campaign | D=social_media')

traffic_source
email_campaign    0.0045
social_media      0.0045
paid_search       0.0045
referral          0.0045
organic_search    0.0045
direct            0.0045

Lowest bounce_rate: email_campaign (0.0045)
  → A=organic_search | B=paid_search | C=email_campaign | D=social_media


---
## Q5 — % dòng order_items.csv có promo_id không null

> Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi
> (tức là promo_id không null) xấp xỉ là bao nhiêu?
>
> A) 12%  B) 25%  C) 39%  D) 54%

In [6]:
total_rows = len(order_items)
promo_rows = order_items['promo_id'].notna().sum()
pct = promo_rows / total_rows * 100

print(f'Total rows   : {total_rows:,}')
print(f'Rows w/ promo: {promo_rows:,}')
print(f'Percentage   : {pct:.1f}%')
print(f'  → A=12% | B=25% | C=39% | D=54%')

Total rows   : 714,669
Rows w/ promo: 276,316
Percentage   : 38.7%
  → A=12% | B=25% | C=39% | D=54%


---
## Q6 — age_group có avg đơn/khách hàng cao nhất (loại null)

> Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào
> có số đơn hàng trung bình trên mỗi khách hàng cao nhất?
>
> A) 55+  B) 25–34  C) 35–44  D) 45–54

In [7]:
# Orders per customer
orders_per_cust = orders.groupby('customer_id').size().reset_index(name='order_count')

# Join with customer age_group (non-null only)
cust_age = customers[customers['age_group'].notna()][['customer_id', 'age_group']]
merged = cust_age.merge(orders_per_cust, on='customer_id', how='left')
merged['order_count'] = merged['order_count'].fillna(0)

avg_orders = (
    merged.groupby('age_group')['order_count']
    .mean()
    .sort_values(ascending=False)
)
print(avg_orders.round(4).to_string())
print(f'\nHighest avg orders: {avg_orders.idxmax()} ({avg_orders.max():.4f})')
print(f'  → A=55+ | B=25–34 | C=35–44 | D=45–54')

age_group
55+      5.4069
45-54    5.3572
35-44    5.3373
25-34    5.2452
18-24    5.2267

Highest avg orders: 55+ (5.4069)
  → A=55+ | B=25–34 | C=35–44 | D=45–54


---
## Q7 — Vùng (region) có tổng doanh thu cao nhất

> Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
> sales_train.csv?
>
> A) West  B) Central  C) East  D) Cả ba vùng xấp xỉ bằng nhau

_Chú ý: sales.csv chỉ có doanh thu tổng theo ngày. Để phân tách theo vùng, ta tính revenue
theo đơn hàng: order_items → orders (zip) → geography (region)._

In [8]:
# Line-item revenue: quantity * unit_price - discount_amount
order_items['line_revenue'] = (
    order_items['quantity'] * order_items['unit_price'] - order_items['discount_amount']
)

# Map order_id → zip via orders
oi_zip = order_items.merge(orders[['order_id', 'zip']], on='order_id', how='left')

# Map zip → region
oi_region = oi_zip.merge(geography[['zip', 'region']], on='zip', how='left')

revenue_by_region = (
    oi_region.groupby('region')['line_revenue']
    .sum()
    .sort_values(ascending=False)
)
print(revenue_by_region.map(lambda x: f'{x:,.0f}').to_string())
print(f'\nHighest revenue: {revenue_by_region.idxmax()}')

# Check if "roughly equal" (D) – compare max vs min ratio
ratio = revenue_by_region.max() / revenue_by_region.min()
print(f'Max/Min ratio: {ratio:.2f}x  (D applies if ≈ 1.0)')
print(f'  → A=West | B=Central | C=East | D=Cả ba xấp xỉ bằng nhau')

region
East       7,291,150,819
Central    4,719,491,268
West       3,670,227,178

Highest revenue: East
Max/Min ratio: 1.99x  (D applies if ≈ 1.0)
  → A=West | B=Central | C=East | D=Cả ba xấp xỉ bằng nhau


---
## Q8 — Phương thức thanh toán nhiều nhất ở đơn cancelled

> Trong các đơn hàng có order_status = 'cancelled' trong orders.csv, phương thức
> thanh toán nào được sử dụng nhiều nhất?
>
> A) credit_card  B) cod  C) paypal  D) bank_transfer

In [9]:
cancelled = orders[orders['order_status'] == 'cancelled']

pm_counts = cancelled['payment_method'].value_counts()
print(pm_counts.to_string())
print(f'\nMost common: {pm_counts.idxmax()} ({pm_counts.max():,})')
print(f'  → A=credit_card | B=cod | C=paypal | D=bank_transfer')

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535

Most common: credit_card (28,452)
  → A=credit_card | B=cod | C=paypal | D=bank_transfer


---
## Q9 — Size (S/M/L/XL) có return rate cao nhất

> Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng
> cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong
> order_items (join với products theo product_id)?
>
> A) S  B) M  C) L  D) XL

In [10]:
SIZES = ['S', 'M', 'L', 'XL']

# Returns per size
ret_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
returns_count = ret_size[ret_size['size'].isin(SIZES)].groupby('size').size()

# Order items per size
oi_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
items_count = oi_size[oi_size['size'].isin(SIZES)].groupby('size').size()

return_rate = (returns_count / items_count).reindex(SIZES).sort_values(ascending=False)
print(return_rate.map(lambda x: f'{x:.4f} ({x*100:.2f}%)').to_string())
print(f'\nHighest return rate: {return_rate.idxmax()}')
print(f'  → A=S | B=M | C=L | D=XL')

size
S     0.0565 (5.65%)
L     0.0562 (5.62%)
M     0.0557 (5.57%)
XL    0.0552 (5.52%)

Highest return rate: S
  → A=S | B=M | C=L | D=XL


---
## Q10 — Kế hoạch trả góp có avg payment_value/đơn cao nhất

> Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
> mỗi đơn hàng cao nhất?
>
> A) 1 kỳ (trả một lần)  B) 3 kỳ  C) 6 kỳ  D) 12 kỳ

In [11]:
avg_by_installment = (
    payments.groupby('installments')['payment_value']
    .mean()
    .sort_values(ascending=False)
)
print(avg_by_installment.round(2).to_string())
print(f'\nHighest avg payment_value: installments={avg_by_installment.idxmax()} ({avg_by_installment.max():.2f})')
print(f'  → A=1 kỳ | B=3 kỳ | C=6 kỳ | D=12 kỳ')

installments
6     24446.65
3     24399.64
12    24245.77
1     24113.27
2       708.47

Highest avg payment_value: installments=6 (24446.65)
  → A=1 kỳ | B=3 kỳ | C=6 kỳ | D=12 kỳ
